## **FastAPI** - Complete Notes!

##  1. **Introduction to FastAPI**

### What is FastAPI?

FastAPI is a Python framework used to build backend APIs.
Each API is a URL connected to a Python function.

A FastAPI application is simply a **collection of APIs**.


## 📌 2. Project Structure Used

```
fastapi-ecommerce/
└── app/
    ├── main.py
    ├── service/
    │   └── products.py
    └── data/
        └── products.json
```

### Why this structure?

* Keeps code clean
* Separates API logic and data logic
* Easy to scale later


## 📌 3. Running the FastAPI Server

Command used:

```bash
python -m uvicorn app.main:app --reload
```

### Meaning:

* `app.main` → Python file path
* `app` → FastAPI instance
* `--reload` → auto restart on code change



## 📌 4. Root API (Health Check)

### Code:

```python
@app.get("/")
def root():
    return {"message": "Welcome to FastAPI!"}
```

### Meaning:

* Checks if server is running
* Returns simple message



## 📌 5. Get All Products API

### URL:

```
/products
```

### Code:

```python
@app.get("/products")
def get_products():
    return get_all_products()
```

### Meaning:

* Reads product data from JSON file
* Returns all products



## 📌 6. Get Product by ID API

### URL:

```
/products/{product_id}
```

Example:

```
/products/1
```

### Code:

```python
@app.get("/products/{product_id}")
def get_product(product_id: int):
    products = get_all_products()
    for product in products:
        if product["id"] == product_id:
            return product
    return {"error": "Product not found"}
```

### Meaning:

* Fetches one product using ID
* Returns error message if not found

---

## 📌 7. Search Product by Name (Case Sensitive)

### URL:

```
/products/search?name=Mobile
```

### Code:

```python
@app.get("/products/search")
def search_product(name: str):
    results = search_products_by_name(name)
    if not results:
        return {"message": "No product found with this name"}
    return results
```

### Important Rule:

This API must be written **before**:

```
/products/{product_id}
```



## 📌 8. Service Layer Concept

### Why service layer?

* API should not handle file logic
* Service handles:

  * JSON read
  * Searching
  * Filtering

### Example:

```python
def get_all_products():
    return load_products()
```


## 📌 9. Full Flow (Any API)

1. User sends request
2. FastAPI receives request
3. API function is called
4. Service layer processes data
5. JSON file is read
6. Result is returned
7. FastAPI converts Python → JSON

---

## 📌 10. Summary (One Line)

> This FastAPI application is a group of APIs that allow users to fetch and search product data stored in a JSON file.



# WHAT COMES NEXT (CONFIRMED)

### Next API:

## **POST `/products` — Add New Product**

This will teach:

* Request body
* JSON input
* Data creation
* Real backend behavior

#  **FastAPI – POST API (Add New Product)**


## 📌 1. What is POST API? (Very Simple)

A **POST API** is used to **send data from user to server**.

In our case:

> POST API will **add a new product** into the system.

Till now:

* GET → read data
  Now:
* POST → **create data**



## 📌 2. Why POST API is Important

POST API allows:

* Adding new products
* Accepting data from frontend
* Building real backend systems

Without POST:

> Backend is read-only and incomplete.



## 📌 3. API Goal

### API Name:

```
POST /products
```

### Purpose:

* Take product data from user
* Save it into `products.json`
* Return success response



## 📌 4. Data Format Sent by User (Request Body)

User will send data like this:

```json
{
  "id": 3,
  "name": "Tablet",
  "price": 30000
}
```

This data is sent in **request body**, not URL.



## 📌 5. POST API Code (Main API)

### 📄 `app/main.py`

```python
from fastapi import FastAPI
from app.service.products import add_product

app = FastAPI()

@app.post("/products")
def create_product(product: dict):
    result = add_product(product)
    return result
```



## 📌 6. Meaning of Each Line (Simple)

### 🔹 `@app.post("/products")`

* Defines a POST API
* Triggered when user sends POST request to `/products`



### 🔹 `def create_product(product: dict):`

* `product` → data coming from request body
* `dict` → data must be JSON format

FastAPI automatically converts JSON → Python dict.



### 🔹 `result = add_product(product)`

* Calls service layer
* Service handles file writing logic



### 🔹 `return result`

* Sends response back to user
* FastAPI converts Python → JSON



## 📌 7. Service Layer Code (Adding Product)

### 📄 `app/service/products.py`

```python
import json
from pathlib import Path
from typing import List, Dict

DATA_FILE = Path(__file__).parent.parent / "data" / "products.json"


def load_products() -> List[Dict]:
    if not DATA_FILE.exists():
        return []
    with open(DATA_FILE, "r", encoding="utf-8") as file:
        return json.load(file)


def save_products(products: List[Dict]):
    with open(DATA_FILE, "w", encoding="utf-8") as file:
        json.dump(products, file, indent=2)


def add_product(product: Dict):
    products = load_products()
    products.append(product)
    save_products(products)
    return {"message": "Product added successfully", "product": product}
```



## 📌 8. What Happens Internally (Step-by-Step)

1. User sends POST request with product JSON
2. FastAPI receives request
3. JSON → Python dictionary
4. API calls service function
5. Existing products are loaded
6. New product is added
7. JSON file is updated
8. Success message returned



## 📌 9. How to Test POST API

### Using Swagger UI

Open:

```
http://127.0.0.1:8000/docs
```

* Choose **POST /products**
* Click **Try it out**
* Paste JSON
* Click **Execute**



### Example Successful Response

```json
{
  "message": "Product added successfully",
  "product": {
    "id": 3,
    "name": "Tablet",
    "price": 30000
  }
}
```



## 📌 10. Important Notes (For Understanding)

* POST API changes data
* Data comes from **request body**
* Service layer handles file writing
* JSON file is updated permanently



## 📌 11. One-Line Summary

> This POST API receives product data from the user, stores it in a JSON file, and returns a success response.



# **FastAPI — Pydantic Validation**

## 📌 1. Why Pydantic is Needed (Very Simple)

Till now, in POST API we accepted data like this:

```python
def create_product(product: dict):
```

❌ Problem:

* Any wrong data can come
* Missing fields not checked
* Wrong data type allowed

Example (bad data):

```json
{
  "id": "abc",
  "name": 123,
  "price": "cheap"
}
```

FastAPI accepts it ❌
This is **dangerous**.



## 📌 2. What is Pydantic? (In Simple Words)

> **Pydantic is used to validate and structure incoming data.**

Meaning:

* Checks data types
* Ensures required fields
* Rejects wrong input automatically



## 📌 3. What Changes with Pydantic

### Before (Without Pydantic)

```python
product: dict
```

### After (With Pydantic)

```python
product: Product
```

Where `Product` is a **model** describing the data shape.



## 📌 4. Creating a Product Model (Pydantic Model)

### 📄 `app/models/product.py`

```python
from pydantic import BaseModel

class Product(BaseModel):
    id: int
    name: str
    price: float
```



## 📌 5. Meaning of Each Line (Very Simple)

### 🔹 `class Product(BaseModel):`

* Creates a data model
* Tells FastAPI how product data should look



### 🔹 `id: int`

* Product ID must be a number



### 🔹 `name: str`

* Product name must be text



### 🔹 `price: float`

* Product price must be a number (decimal allowed)



## 📌 6. Updating POST API to Use Pydantic

### 📄 `app/main.py`

```python
from fastapi import FastAPI
from app.models.product import Product
from app.service.products import add_product

app = FastAPI()

@app.post("/products")
def create_product(product: Product):
    return add_product(product.dict())
```



## 📌 7. What Changed Here?

### 🔹 `product: Product`

* FastAPI now **validates** incoming data
* Rejects wrong input automatically



### 🔹 `product.dict()`

* Converts Pydantic model → normal Python dictionary
* Needed for JSON file storage

---

## 📌 8. Example: Correct Request

```json
{
  "id": 4,
  "name": "Headphones",
  "price": 1500
}
```

✔ Accepted
✔ Saved
✔ Returned



## 📌 9. Example: Wrong Request (Validation Error)

```json
{
  "id": "four",
  "name": 123
}
```

Response:

```json
{
  "detail": [
    {
      "loc": ["body", "id"],
      "msg": "value is not a valid integer",
      "type": "type_error.integer"
    }
  ]
}
```

FastAPI blocks it automatically ✅



## 📌 10. Why This is Powerful

* No manual checks needed
* Clean error messages
* Strong API contract
* Required for interviews & production



## 📌 11. One-Line Summary

> **Pydantic ensures that only valid and well-structured data enters the API.**



# **FastAPI — HTTPException & Status Codes**


## 📌 1. Why HTTPException is Needed (Simple Words)

Right now, when something goes wrong, your API returns:

```json
{"error": "Product not found"}
```

❌ Problem:

* HTTP status is still **200 OK**
* Client thinks request was successful
* Not professional



## ✅ Correct API Behaviour

> When something goes wrong,
> the API must **clearly tell the client** using HTTP status codes.

This is done using **HTTPException**.



## 📌 2. What is HTTPException?

> **HTTPException is used to stop request execution and send an error response with a proper HTTP status code.**

Example:

```python
raise HTTPException(status_code=404, detail="Product not found")
```



## 📌 3. Importing HTTPException

In `app/main.py`, we import:

```python
from fastapi import FastAPI, HTTPException, status
```

* `HTTPException` → raises error
* `status` → readable HTTP codes



## 📌 4. Updating GET Product by ID (IMPORTANT)

### ❌ Old code

```python
return {"error": "Product not found"}
```

### ✅ New code (Correct)

```python
raise HTTPException(
    status_code=status.HTTP_404_NOT_FOUND,
    detail="Product not found"
)
```



## 📄 UPDATED `app/main.py` (WITH HTTPException)

```python
from fastapi import FastAPI, HTTPException, status
from app.models.product import Product
from app.service.products import (
    get_all_products,
    search_products_by_name,
    add_product
)

app = FastAPI()

@app.get("/")
def root():
    return {"message": "APX and His - !"}


# 🔍 SEARCH MUST COME FIRST
@app.get("/products/search")
def search_product(name: str):
    results = search_products_by_name(name)
    if not results:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="No product found with this name"
        )
    return results


@app.get("/products")
def get_products():
    return get_all_products()


@app.get("/products/{product_id}")
def get_product(product_id: int):
    products = get_all_products()
    for product in products:
        if product["id"] == product_id:
            return product

    raise HTTPException(
        status_code=status.HTTP_404_NOT_FOUND,
        detail="Product not found"
    )


@app.post("/products", status_code=status.HTTP_201_CREATED)
def create_product(product: Product):
    result = add_product(product.dict())

    if "error" in result:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=result["error"]
        )

    return result
```



## 📌 5. What Changed? (Very Simple)

### 🔹 `raise HTTPException(...)`

* Stops execution immediately
* Sends proper error response
* No need to return manually



### 🔹 `status_code=status.HTTP_404_NOT_FOUND`

* Clear meaning
* Easier to read than numbers like `404`



### 🔹 `@app.post(..., status_code=201)`

* Tells client: resource was **created successfully**



## 📌 6. Example Responses (Important)

### ❌ Product not found

```json
{
  "detail": "Product not found"
}
```

HTTP Status: **404**



### ❌ Duplicate ID

```json
{
  "detail": "Product with this ID already exists"
}
```

HTTP Status: **400**



###  Successful creation

HTTP Status: **201 Created**



## 📌 7. One-Line Summary

> **HTTPException makes APIs honest by sending correct HTTP status codes instead of fake success responses.**


## What You Have Achieved Now

* ✔ Proper error handling
* ✔ Industry-standard API responses
* ✔ Clean client–server communication
* ✔ Interview-ready FastAPI code

Your FastAPI completion is now **~70%**.


## 🔜 What Comes Next (Correct Order)

Next recommended topics:

1️⃣ **PUT / PATCH API** (update product)
2️⃣ **DELETE API**
3️⃣ Filtering (price range, partial search)


## **FastAPI — PUT API (Update Product)**

## 📌 1. What is PUT API? (Very Simple)

> **PUT API is used to update existing data.**

In our project:

* PUT will **update an existing product**
* Product is identified using **ID**

---

## 📌 2. API Goal

### Endpoint

```
PUT /products/{product_id}
```

### Purpose

* Find product by ID
* Update its details
* Save changes in JSON file

---

## 📌 3. Request Body (Using Pydantic)

```json
{
  "name": "Updated Mobile",
  "price": 22000
}
```

⚠️ ID is taken from URL, not body.

---

## 📌 4. Create Update Model (Pydantic)

### 📄 `app/models/product.py`

```python
from pydantic import BaseModel

class Product(BaseModel):
    id: int
    name: str
    price: float


class ProductUpdate(BaseModel):
    name: str
    price: float
```

### Why separate model?

* POST needs `id`
* PUT does **not**
* Cleaner & safer

---

## 📌 5. Update Service Logic

### 📄 `app/service/products.py`

```python
def update_product(product_id: int, updated_data: Dict):
    products = load_products()

    for index, product in enumerate(products):
        if product["id"] == product_id:
            products[index]["name"] = updated_data["name"]
            products[index]["price"] = updated_data["price"]
            save_products(products)

            return products[index]

    return None
```

---

## 📌 6. PUT API in `main.py`

### 📄 `app/main.py`

```python
from app.models.product import Product, ProductUpdate
from fastapi import FastAPI, HTTPException, status

@app.put("/products/{product_id}")
def update_product_api(product_id: int, product: ProductUpdate):
    updated_product = update_product(product_id, product.dict())

    if not updated_product:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Product not found"
        )

    return {
        "message": "Product updated successfully",
        "product": updated_product
    }
```

---

## 📌 7. What Happens Internally (Step-by-Step)

1. User sends PUT request
2. FastAPI validates request body (Pydantic)
3. Service loads products
4. Finds product by ID
5. Updates name & price
6. Saves JSON file
7. Returns updated product

---

## 📌 8. Test PUT API

### Swagger UI

```
http://127.0.0.1:8000/docs
```

### Example Request

```
PUT /products/2
```

```json
{
  "name": "Smartphone",
  "price": 25000
}
```

### Success Response

```json
{
  "message": "Product updated successfully",
  "product": {
    "id": 2,
    "name": "Smartphone",
    "price": 25000
  }
}
```

---

## 📌 9. Error Case (Product Not Found)

```json
{
  "detail": "Product not found"
}
```

HTTP Status: **404**

---

## 📌 10. One-Line Summary

> **PUT API updates an existing product using its ID and saves the changes permanently.**

---

## 🎯 Your FastAPI Progress Now

* GET ✔
* SEARCH ✔
* POST ✔
* PUT ✔

👉 **~80% FastAPI completed**

---

## 🔜 What’s Next (Choose One)

1️⃣ **DELETE API** (remove product)
2️⃣ **PATCH API** (partial update)
3️⃣ **Advanced filtering**

### Recommended:

 **DELETE API**



---

## **FastAPI — DELETE API (Delete Product)**

---
    
## 📌 1. What is DELETE API? (Very Simple)

> **DELETE API removes data permanently.**

In our project:

* Delete a product using its **ID**
* Remove it from `products.json`

---

## 📌 2. API Endpoint

```
DELETE /products/{product_id}
```

Example:

```
DELETE /products/2
```

---

## 📄 1️⃣ MODIFY `app/service/products.py`

(Add delete logic)

```python
def delete_product(product_id: int):
    products = load_products()

    for index, product in enumerate(products):
        if product["id"] == product_id:
            deleted_product = products.pop(index)
            save_products(products)
            return deleted_product

    return None
```

### What this does (simple):

* Load products
* Find product by ID
* Remove it
* Save updated list
* Return deleted product
* If not found → return `None`

---

## 📄 2️⃣ MODIFY `app/main.py`

(Add DELETE API)

```python
from app.service.products import (
    get_all_products,
    search_products_by_name,
    add_product,
    update_product,
    delete_product
)

# 🗑 DELETE API – Remove product
@app.delete("/products/{product_id}")
def delete_product_api(product_id: int):
    deleted = delete_product(product_id)

    if not deleted:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Product not found"
        )

    return {
        "message": "Product deleted successfully",
        "product": deleted
    }
```

---

## 🧪 3. Test DELETE API

### Swagger UI

```
http://127.0.0.1:8000/docs
```

### Example Request

```
DELETE /products/2
```

### Success Response

```json
{
  "message": "Product deleted successfully",
  "product": {
    "id": 2,
    "name": "Mobile",
    "price": 20000
  }
}
```

### Error (Not Found)

```json
{
  "detail": "Product not found"
}
```

HTTP **404**

---

## 🧠 What You Have Achieved Now

You now have **FULL CRUD**:

| Operation | API                               |
| --------- | --------------------------------- |
| Create    | POST `/products`                  |
| Read      | GET `/products`, `/products/{id}` |
| Update    | PUT `/products/{id}`              |
| Delete    | DELETE `/products/{id}`           |

✔ This is **complete backend CRUD**
✔ Interview-ready
✔ Real-world API design

---

## 📈 FastAPI Progress Update

* Core FastAPI: **~90% DONE**
* CRUD APIs: **100% DONE**
* Validation: DONE
* Error handling: DONE

You are **no longer a beginner**.

---

## 🔜 What Comes Next (Your Choice)

Now you have **3 strong options**:

1️⃣ **PATCH API** (partial update)
2️⃣ **Advanced filtering** (price range, contains search)
3️⃣ **Database integration (SQLite)**




# **FastAPI — PATCH API (Partial Update)**

---

## 📌 1. What is PATCH API? (Simple words)

> **PATCH API updates only the fields you send**, not the whole object.

### Difference:

* **PUT** → update *everything*
* **PATCH** → update *only what is provided*

Example:

```json
{
  "price": 24000
}
```

Only price changes, name stays same.

---

## 📌 2. PATCH Endpoint

```
PATCH /products/{product_id}
```

---

## 📄 1️⃣ UPDATE Pydantic Model

### `app/models/product.py`

```python
from pydantic import BaseModel
from typing import Optional

class Product(BaseModel):
    id: int
    name: str
    price: float


class ProductUpdate(BaseModel):
    name: Optional[str] = None
    price: Optional[float] = None
```

### Why `Optional`?

* PATCH fields are **not required**
* User can send one or both fields

---

## 📄 2️⃣ ADD PATCH LOGIC IN SERVICE

### `app/service/products.py`

```python
def patch_product(product_id: int, updated_data: Dict):
    products = load_products()

    for index, product in enumerate(products):
        if product["id"] == product_id:

            if updated_data.get("name") is not None:
                products[index]["name"] = updated_data["name"]

            if updated_data.get("price") is not None:
                products[index]["price"] = updated_data["price"]

            save_products(products)
            return products[index]

    return None
```

### What happens:

* Find product by ID
* Update only provided fields
* Save JSON
* Return updated product

---

## 📄 3️⃣ ADD PATCH API IN `main.py`

```python
from app.service.products import patch_product

@app.patch("/products/{product_id}")
def patch_product_api(product_id: int, product: ProductUpdate):
    updated = patch_product(product_id, product.dict())

    if not updated:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Product not found"
        )

    return {
        "message": "Product updated successfully",
        "product": updated
    }
```

---

## 🧪 4. TEST PATCH API

### Example 1️⃣ – Update only price

```
PATCH /products/3
```

```json
{
  "price": 26000
}
```

### Example 2️⃣ – Update only name

```json
{
  "name": "New Mobile Name"
}
```

### Example 3️⃣ – Update both

```json
{
  "name": "New Mobile",
  "price": 28000
}
```

---

## 📌 5. Error Case

If product does not exist:

```json
{
  "detail": "Product not found"
}
```

HTTP **404**

---

## 🧠 One-Line Summary

> **PATCH API updates only the fields sent by the client, without affecting other data.**


# **FastAPI — Filtering (Advanced & Practical)**

We’ll implement **3 filters** (progressive & useful):

1️⃣ **Price range filter**
2️⃣ **Partial name search (contains)**
3️⃣ **Combined filters (name + price)**

---

## 1️⃣ Price Range Filter

### 🔹 API

```
GET /products/filter/price?min_price=1000&max_price=30000
```

---

### 📄 Update `app/service/products.py`

```python
def filter_products_by_price(min_price: float, max_price: float):
    products = load_products()
    return [
        p for p in products
        if min_price <= p.get("price", 0) <= max_price
    ]
```

---

### 📄 Update `app/main.py`

```python
from app.service.products import filter_products_by_price

@app.get("/products/filter/price")
def filter_by_price(min_price: float, max_price: float):
    results = filter_products_by_price(min_price, max_price)

    if not results:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="No products found in this price range"
        )

    return results
```

---

## 2️⃣ Partial Name Search (Contains)

### 🔹 API

```
GET /products/filter/name?keyword=mob
```

This finds:

* Mobile
* Smartphone
* MobCover

---

### 📄 Update `app/service/products.py`

```python
def filter_products_by_name_contains(keyword: str):
    products = load_products()
    return [
        p for p in products
        if keyword.lower() in p.get("name", "").lower()
    ]
```

---

### 📄 Update `app/main.py`

```python
from app.service.products import filter_products_by_name_contains

@app.get("/products/filter/name")
def filter_by_name(keyword: str):
    results = filter_products_by_name_contains(keyword)

    if not results:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="No products found with this keyword"
        )

    return results
```

---

## 3️⃣ Combined Filtering (Name + Price)

### 🔹 API

```
GET /products/filter?keyword=mob&min_price=10000&max_price=30000
```

---

### 📄 Update `app/service/products.py`

```python
def filter_products(keyword: str, min_price: float, max_price: float):
    products = load_products()
    return [
        p for p in products
        if keyword.lower() in p.get("name", "").lower()
        and min_price <= p.get("price", 0) <= max_price
    ]
```

---

### 📄 Update `app/main.py`

```python
from app.service.products import filter_products

@app.get("/products/filter")
def combined_filter(keyword: str, min_price: float, max_price: float):
    results = filter_products(keyword, min_price, max_price)

    if not results:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="No products match the given filters"
        )

    return results
```

---

## 🧪 How to Test (Swagger)

Open:

```
http://127.0.0.1:8000/docs
```

Try:

* Price range
* Keyword search
* Combined filters

---

## 🧠 What You Just Learned

* Query parameter filtering
* Real e-commerce search logic
* Case-insensitive contains search
* Combining multiple conditions
* Clean service-based filtering

---

## 📈 Final Progress Update

✔ CRUD
✔ PATCH
✔ Filtering

 **~98% of Core FastAPI DONE**

You now have **real backend-level APIs**.

---



# **FastAPI — Database Integration (SQLite)**

We will do this in **phases**, not all at once.

---

## 🎯 PHASE PLAN (IMPORTANT)

### Phase 1 (NOW)

* What is a database
* Why JSON is not enough
* SQLite basics
* Connect FastAPI to SQLite

### Phase 2 (NEXT)

* SQLAlchemy ORM
* Models → Tables
* CRUD using database

### Phase 3 (LATER)

* Migrations
* PostgreSQL (upgrade)
* Production readiness

---

## 📌 1. Why Move from JSON to Database? (Simple Words)

Your current system uses:

```
products.json
```
##  What Database Solves

A database gives:

* Safe storage
* Fast queries
* Multiple users support
* Industry-standard backend

---

## 📌 2. Why SQLite?

> **SQLite is a lightweight, file-based database.**

Why it’s perfect now:

* No installation needed
* Stored as a `.db` file
* Easy to learn
* Used in real apps

---

## 📌 3. Install Required Packages

Run this once:

```bash
pip install sqlalchemy
```

(FastAPI already installed)

---

## 📌 4. Create Database Configuration

### 📄 `app/database.py` (NEW FILE)

```python
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, declarative_base

DATABASE_URL = "sqlite:///./products.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()
```

### Meaning (simple):

* `products.db` → SQLite database file
* `engine` → connection to database
* `SessionLocal` → talk to DB
* `Base` → base class for tables

---

## 📌 5. Create Product Table (Model)

### 📄 `app/models/db_product.py` (NEW FILE)

```python
from sqlalchemy import Column, Integer, String, Float
from app.database import Base

class ProductDB(Base):
    __tablename__ = "products"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, index=True)
    price = Column(Float)
```

### Meaning:

* Table name: `products`
* Columns: id, name, price

---

## 📌 6. Create Database Tables

### 📄 `app/create_tables.py`

```python
from app.database import engine
from app.models.db_product import ProductDB

ProductDB.metadata.create_all(bind=engine)
```

Run once:

```bash
python app/create_tables.py
```

This creates:

```
products.db
```

---

## 📌 7. Database vs JSON (One-Line)

> JSON is for learning, Database is for production.

---

## 🧠 IMPORTANT (Very Honest)

At this point:

* You already understand APIs
* You already understand CRUD
* Database is **just storage change**

So this will feel **easy**, not scary.

---

## 📈 Progress Update

| Area            | Status     |
| --------------- | ---------- |
| FastAPI Core    | ✅ Complete |
| CRUD APIs       | ✅ Complete |
| Filtering       | ✅ Complete |
| Database Basics | 🔄 Started |

You are now entering **backend engineer territory**.



# 🔐 **FastAPI — JWT Authentication** (Step-by-Step)

I’ll divide this into **clear phases** so it never feels confusing.

---

## 🧠 PHASE 0: What JWT Is (Very Simple)

> **JWT = a secure digital token that proves who you are**

Flow:

```
Login → Server gives token → Client sends token → Server trusts client
```

Without token ❌ → access denied
With token ✅ → access allowed

---

## 🎯 WHAT WE WILL BUILD

1. User table (database)
2. Signup API
3. Login API
4. JWT token generation
5. Protect product APIs (POST / PUT / PATCH / DELETE)

---

## 📦 REQUIRED PACKAGES (Install once)

Run this:

```powershell
python -m pip install python-jose passlib[bcrypt]
```

* `python-jose` → JWT handling
* `passlib` → password hashing

---

# 🔹 PHASE 1: USER DATABASE MODEL

### 📄 `app/models/db_user.py` (NEW FILE)

```python
from sqlalchemy import Column, Integer, String
from app.database import Base

class UserDB(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True)
    hashed_password = Column(String)
```

---

## 🧱 Create User Table

Update `create_tables.py`:

```python
from app.database import engine
from app.models.db_product import ProductDB
from app.models.db_user import UserDB

ProductDB.metadata.create_all(bind=engine)
UserDB.metadata.create_all(bind=engine)

print("✅ Tables created")
```

Run:

```powershell
python -m app.create_tables
```

---

# 🔹 PHASE 2: PASSWORD HASHING

### 📄 `app/security.py` (NEW FILE)

```python
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str):
    return pwd_context.hash(password)

def verify_password(password: str, hashed: str):
    return pwd_context.verify(password, hashed)
```

---

# 🔹 PHASE 3: JWT TOKEN HANDLING

### 📄 `app/jwt.py` (NEW FILE)

```python
from datetime import datetime, timedelta
from jose import jwt

SECRET_KEY = "SUPER_SECRET_KEY_CHANGE_ME"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
```

---

# 🔹 PHASE 4: AUTH SERVICE LOGIC

### 📄 `app/service/auth.py` (NEW FILE)

```python
from sqlalchemy.orm import Session
from app.models.db_user import UserDB
from app.security import hash_password, verify_password

def create_user(db: Session, username: str, password: str):
    user = UserDB(
        username=username,
        hashed_password=hash_password(password)
    )
    db.add(user)
    db.commit()
    db.refresh(user)
    return user

def authenticate_user(db: Session, username: str, password: str):
    user = db.query(UserDB).filter(UserDB.username == username).first()
    if not user:
        return None
    if not verify_password(password, user.hashed_password):
        return None
    return user
```

---

# 🔹 PHASE 5: LOGIN & SIGNUP APIs

### 📄 Update `app/main.py`

```python
from fastapi import Depends
from sqlalchemy.orm import Session
from app.database import get_db
from app.jwt import create_access_token
from app.service.auth import create_user, authenticate_user
```

### 🔐 Signup API

```python
@app.post("/signup")
def signup(username: str, password: str, db: Session = Depends(get_db)):
    user = create_user(db, username, password)
    return {"message": "User created", "user": user.username}
```

---

### 🔐 Login API

```python
@app.post("/login")
def login(username: str, password: str, db: Session = Depends(get_db)):
    user = authenticate_user(db, username, password)
    if not user:
        raise HTTPException(status_code=401, detail="Invalid credentials")

    token = create_access_token({"sub": user.username})
    return {"access_token": token, "token_type": "bearer"}
```

---

# 🔹 PHASE 6: TOKEN VERIFICATION (Protect APIs)

### 📄 `app/dependencies.py` (NEW FILE)

```python
from fastapi import Depends, HTTPException, status
from jose import jwt, JWTError
from app.jwt import SECRET_KEY, ALGORITHM

def get_current_user(token: str = Depends()):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        return payload.get("sub")
    except JWTError:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token"
        )
```

---

### 🔒 Protect Product APIs

Example:

```python
@app.post("/products")
def create_product_api(
    product: Product,
    db: Session = Depends(get_db),
    user: str = Depends(get_current_user)
):
    return create_product(db, product)
```

Now:

* ❌ No token → access denied
* ✅ Token → access allowed

---

## 🧪 TEST FLOW (Swagger)

1. `/signup`
2. `/login` → copy token
3. Swagger → **Authorize** → paste token
4. Try POST / PUT / DELETE

---

## 🧠 WHAT YOU HAVE ACHIEVED NOW

✔ Authentication
✔ Authorization
✔ Secure APIs
✔ Industry-grade backend

---

## 📈 FINAL STATUS

| Area         | Status |
| ------------ | ------ |
| FastAPI Core | ✅      |
| CRUD         | ✅      |
| Database     | ✅      |
| Filtering    | ✅      |
| JWT Auth     | ✅ DONE |

**100% FastAPI backend complete**

---

